In [2]:
pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 6.8 MB/s eta 0:00:00


In [35]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import numpy as np
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import os
from io import BytesIO

In [36]:
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id="c90ea70f713742c69defa9541f372d06",
    client_secret="bfa4e194648b47d6a84da1ba3f9026b3"
))

In [62]:
def get_artist_info(artist_name):
    result = sp.search(q=f"artist:{artist_name}", type="artist", limit=1)
    if not result['artists']['items']:
        return None

    artist = result['artists']['items'][0]
    artist_id = artist['id']
    followers = artist['followers']['total']
    popularity = artist['popularity']
    image_url = artist['images'][0]['url'] if artist['images'] else None

    top_track_data = sp.artist_top_tracks(artist_id, country='US')
    top_track = top_track_data['tracks'][0] if top_track_data['tracks'] else None

    return {
        "id": artist_id,
        "name": artist['name'],
        "followers": followers,
        "popularity": popularity,
        "image_url": image_url,
        "top_track_name": top_track['name'] if top_track else "N/A",
        "top_track_url": top_track['external_urls']['spotify'] if top_track else None,
        "top_track_preview": top_track['preview_url'] if top_track else None
    }

In [163]:
def simulate_model():
    X = np.array([
        [5000, 20],     # not mainstream
        [20000, 40],    # not mainstream
        [500000, 60],   # not mainstream
        [1000000, 70],  # mainstream
        [3000000, 80],  # mainstream
        [10000000, 90]  # mainstream
    ])
    y = [0, 0, 0, 0, 1, 1]
    model = LogisticRegression()
    model.fit(X, y)
    return model


In [202]:
def predict_mainstream_probabilities(model, followers, popularity):
    years = []
    base = np.array([[followers, popularity]])
    if popularity >= 80:
        growth_rate_followers = 0.8
        growth_rate_popularity = 0.8
    elif popularity >50 and popularity <80:
        growth_rate_followers = 1.5
        growth_rate_popularity = 1.5
    else:
        growth_rate_followers = 2.6
        growth_rate_popularity = 2.6
    for year in range(1, 6):
        proj_followers = followers * (growth_rate_followers ** year)
        proj_popularity = min(popularity * (growth_rate_popularity ** year), 100)
        prob = model.predict_proba([[proj_followers, proj_popularity]])[0][1]
        years.append(round(prob, 3))
    return years


In [195]:
def plot_probabilities(probabilities, artist_name):
       years = [f"Year {i+1}" for i in range(len(probabilities))]
       plt.figure(figsize=(6, 4))
       plt.plot(years, probabilities, marker='o')
       plt.ylim(0, 1)
       plt.title(f"Mainstream Prediction for {artist_name}")
       plt.ylabel("Probability")
       plt.grid(True)
       plt.tight_layout()
       plt.show()  # Display the plot directly
       # Remove the lines that save to file